
# 02_SOLVER_v1 — Solver ALOC_REC por Rateio Proporcional

## Registro das regras desta versão

Esta versão **v1** foi criada para diferenciar do módulo anterior baseado em otimização linear (`linprog`).

### Escopo

1. Processa **somente o primeiro `MES_REF`** da base.
2. Otimiza/balanceia **somente por `ALOC_REC`**.
3. `COD_FER_UNID` ainda **não é etapa de solver** nesta versão; ele permanece como informação de roteiro/auditoria e participa da capacidade via `HOR_FER`.
4. A entrada é uma única base: `bd_LTP_NEC_SOLVER.xlsx`.
5. A saída é a planilha `bd_SOLVER_ALOC_REC_V1.xlsx`.

### Regras de demanda e roteiro

6. `NEC_PCS` é a demanda do item no mês.
7. `NEC_PCS` pertence ao nível:
   - `MES_REF`
   - `ID_PROD_UNID_FAT`
8. Roteiros e recursos são formas possíveis de atender o `NEC_PCS`.
9. A ordem das alternativas respeita:
   - menor `PRIOR_MATPAR` primeiro
   - menor `PRIOR_ROT` primeiro
10. Menor prioridade é melhor, mas alternativa não é bloqueante.
11. O saldo não atendido de uma prioridade passa para a próxima alternativa.

### Regra de capacidade

12. A capacidade oficial da alternativa é:

```text
HOR_CAP = min(HOR_REC, HOR_FER)
```

13. Se `HOR_REC = 0` ou `HOR_FER = 0`, então `HOR_CAP = 0`.
14. Alternativas com `HOR_CAP = 0` aparecem na auditoria, mas não entram no plano de produção.
15. Manter sempre nas saídas:
   - `HOR_REC`
   - `HOR_FER`
   - `HOR_CAP`

### Regra de balanceamento / rateio

16. Esta versão **não usa faixa, meta ou percentual mínimo fixo**.
17. Quando uma máquina (`ALOC_REC`) não tem capacidade para atender todos os itens que disputam aquele recurso, o estouro é rateado proporcionalmente.
18. O fator de rateio é:

```text
FATOR_RATEIO_ALOC_REC = HOR_CAP_DISPONIVEL / HR_SOLICITADA_TOTAL_ALOC_REC
```

19. Cada item recebe produção proporcional à sua solicitação naquele recurso/prioridade.
20. Assim, todos os itens que disputam o gargalo “pagam um pouco da conta”, em vez de um item ser zerado arbitrariamente.

### Não atendimento

21. Não atendimento é somente em peças:
   - `QTD_NAO_ATEND_SOLVER`
22. Não calcular gap em horas, porque `PCS_HORA` pertence ao roteiro/recurso, não ao produto.

### Guias de saída

A planilha final deve conter:

1. `01_RESUMO_MAQUINAS`
2. `02_PLANO_PRODUCAO`
3. `03_NAO_ATEND_ITEM`
4. `04_AUDITORIA_CAPACIDADE`
5. `05_ALTERNATIVAS_ITEM`
6. `99_VALIDACAO`

9. A guia `01_RESUMO_MAQUINAS` mantém apenas `QTD_ITENS`; a coluna `QTD_ROTAS` foi removida para evitar duplicidade de leitura no resumo por `ALOC_REC`.


In [1]:

# ============================================================
# 00. CONFIGURAÇÃO
# ============================================================

print("02_SOLVER_v1_RATEIO_PROPORCIONAL_ALOC_REC")

from pathlib import Path
import numpy as np
import pandas as pd
from openpyxl import load_workbook

# Caminho padrão do projeto no Windows
OUTPUT_DIR_WINDOWS = Path(r"C:\Users\carlo\OneDrive\BC\03. Projetos Bedin\01. Krona\LTP\02_OUTPUT")
INPUT_FILE_WINDOWS = OUTPUT_DIR_WINDOWS / "bd_LTP_NEC_SOLVER.xlsx"
OUTPUT_SOLVER_FILE_WINDOWS = OUTPUT_DIR_WINDOWS / "bd_SOLVER_ALOC_REC_V1.xlsx"

# Fallback para execução em ambiente local/sandbox
INPUT_FILE_FALLBACK = Path("/mnt/data/bd_LTP_NEC_SOLVER.xlsx")
OUTPUT_SOLVER_FILE_FALLBACK = Path("/mnt/data/bd_SOLVER_ALOC_REC_V1.xlsx")

if INPUT_FILE_WINDOWS.exists():
    INPUT_FILE = INPUT_FILE_WINDOWS
    OUTPUT_SOLVER_FILE = OUTPUT_SOLVER_FILE_WINDOWS
else:
    INPUT_FILE = INPUT_FILE_FALLBACK
    OUTPUT_SOLVER_FILE = OUTPUT_SOLVER_FILE_FALLBACK

print("INPUT_FILE:", INPUT_FILE)
print("OUTPUT_SOLVER_FILE:", OUTPUT_SOLVER_FILE)

# Flags mantidas do processo anterior
LOTE_MIN_FLAG = True
MULTIPLO_EMB_FLAG = True

TOL = 1e-7


02_SOLVER_v1_RATEIO_PROPORCIONAL_ALOC_REC
INPUT_FILE: /mnt/data/bd_LTP_NEC_SOLVER.xlsx
OUTPUT_SOLVER_FILE: /mnt/data/bd_SOLVER_ALOC_REC_V1.xlsx


In [2]:

# ============================================================
# 01. FUNÇÕES AUXILIARES
# ============================================================

def to_num(s):
    return pd.to_numeric(s, errors="coerce").fillna(0)


def ensure_columns(df: pd.DataFrame, cols, default=0):
    for c in cols:
        if c not in df.columns:
            df[c] = default
    return df


def calcular_nec_pcs_se_necessario(df: pd.DataFrame) -> pd.DataFrame:
    '''Calcula NEC_PCS somente se a coluna não existir na base.

    Regra herdada do processo anterior:
    NEC_PCS = demanda/carteira/previsão/estoque segurança
              - estoques/transferências/origens/triangulações
              + previsão próximo mês quando MESMA_REG = NAO
              + LTP_COMP_NEC_PCS
              respeitando LIMIT_PCS, lote mínimo e múltiplo de embalagem.
    '''
    df = df.copy()

    if "NEC_PCS" in df.columns:
        df["NEC_PCS"] = to_num(df["NEC_PCS"])
        print("NEC_PCS lido da base de entrada.")
        return df

    required = [
        "LTP_CART_ARR_MES_ANT", "LTP_CART_MES_ATUAL", "LTP_SALDO_PREV_PCS", "LTP_EST_SEG_PCS",
        "LTP_EST_INI_PCS", "LTP_EST_TRANS_PCS", "ORI_TOT_PCS", "TRIANG_TOT_PCS",
        "LTP_SALDO_PREV_PROX_MES_PCS", "LTP_COMP_NEC_PCS", "LIMIT_PCS", "LOTE_MIN", "QTD_EMB"
    ]
    df = ensure_columns(df, required, 0)

    for c in required:
        df[c] = to_num(df[c])

    base_comum = (
        df["LTP_CART_ARR_MES_ANT"]
        + df["LTP_CART_MES_ATUAL"]
        + df["LTP_SALDO_PREV_PCS"]
        + df["LTP_EST_SEG_PCS"]
        - df["LTP_EST_INI_PCS"]
        - df["LTP_EST_TRANS_PCS"]
        - df["ORI_TOT_PCS"]
        - df["TRIANG_TOT_PCS"]
    )

    mesma_reg_nao = df.get("MESMA_REG", "SIM").astype(str).str.upper().eq("NAO")

    nec = np.where(
        mesma_reg_nao,
        base_comum + df["LTP_SALDO_PREV_PROX_MES_PCS"],
        base_comum
    )

    nec = pd.Series(nec, index=df.index).clip(lower=0)
    nec = nec + df["LTP_COMP_NEC_PCS"]
    nec = np.maximum(nec, df["LIMIT_PCS"])

    if LOTE_MIN_FLAG:
        mask_lote = (nec > 0) & (df["LOTE_MIN"] > 0)
        nec = np.where(mask_lote, np.maximum(nec, df["LOTE_MIN"]), nec)

    if MULTIPLO_EMB_FLAG:
        tipo = df.get("TIPO_PROD", "").astype(str).str.upper()
        mask_emb = (nec > 0) & (df["QTD_EMB"] > 0) & (tipo.isin(["PA", "MR"]))
        nec = np.where(mask_emb, np.ceil(nec / df["QTD_EMB"]) * df["QTD_EMB"], nec)

    df["NEC_PCS"] = pd.Series(nec, index=df.index).fillna(0)
    print("NEC_PCS calculado internamente porque a base não trouxe a coluna.")
    return df


def preparar_base(df: pd.DataFrame):
    df = df.copy()

    # Numéricos principais
    for c in ["PRIOR_MATPAR", "PRIOR_ROT", "PCS_HORA", "HOR_REC", "HOR_FER", "NEC_PCS"]:
        if c in df.columns:
            df[c] = to_num(df[c])

    # Capacidade oficial da alternativa, sem fallback
    df["HOR_CAP"] = df[["HOR_REC", "HOR_FER"]].min(axis=1)

    # Primeiro mês
    mes_ref = df["MES_REF"].min()
    bd_mes = df[df["MES_REF"].eq(mes_ref)].copy()

    print("MES_REF processado:", mes_ref)
    print("Linhas bd_mes:", len(bd_mes))

    return bd_mes


In [3]:

# ============================================================
# 02. LEITURA DA BASE
# ============================================================

bd = pd.read_excel(INPUT_FILE)
print("Base lida:", bd.shape)
print("Colunas:", len(bd.columns))

bd = calcular_nec_pcs_se_necessario(bd)
bd_mes = preparar_base(bd)


Base lida: (5728, 52)
Colunas: 52
NEC_PCS calculado internamente porque a base não trouxe a coluna.
MES_REF processado: 2026-06-01 00:00:00
Linhas bd_mes: 5728


In [4]:

# ============================================================
# 03. MONTAGEM DA DEMANDA E DAS ALTERNATIVAS
# ============================================================

# Demanda única por item
meta_cols = [
    "MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD",
    "UNID_PROD", "UNID_FAT", "TIPO_PROD"
]

bd_demanda_solver = (
    bd_mes
    .sort_values(["ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT"])
    .groupby(["MES_REF", "ID_PROD_UNID_FAT"], as_index=False)
    .agg({
        **{c: "first" for c in meta_cols if c not in ["MES_REF", "ID_PROD_UNID_FAT"] and c in bd_mes.columns},
        "NEC_PCS": "max"
    })
)

bd_demanda_solver = bd_demanda_solver[bd_demanda_solver["NEC_PCS"] > TOL].copy()

# Alternativas de produção/auditoria
alt_cols = [
    "MES_REF", "ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT",
    "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP",
    "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD",
    "ID_RECURSO", "ID_FERRAMENTA"
]

bd_alternativas_item = bd_mes[[c for c in alt_cols if c in bd_mes.columns]].copy()
bd_alternativas_item = bd_alternativas_item[
    bd_alternativas_item["ID_PROD_UNID_FAT"].isin(bd_demanda_solver["ID_PROD_UNID_FAT"])
].copy()

bd_alternativas_item = bd_alternativas_item.drop_duplicates(
    [c for c in ["MES_REF", "ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "ALOC_REC", "COD_FER_UNID"] if c in bd_alternativas_item.columns]
).reset_index(drop=True)

# Alternativas válidas para produção
bd_alternativas_validas = bd_alternativas_item[
    bd_alternativas_item["ALOC_REC"].notna()
    & (bd_alternativas_item["PCS_HORA"] > 0)
    & (bd_alternativas_item["HOR_CAP"] > 0)
].copy()

print("Itens com demanda:", len(bd_demanda_solver))
print("Alternativas auditadas:", len(bd_alternativas_item))
print("Alternativas válidas para produção:", len(bd_alternativas_validas))
print("Alternativas com HOR_CAP zero:", (bd_alternativas_item["HOR_CAP"] <= 0).sum())


Itens com demanda: 1144
Alternativas auditadas: 2399
Alternativas válidas para produção: 2191
Alternativas com HOR_CAP zero: 206


In [5]:

# ============================================================
# 04. SOLVER V1 - RATEIO PROPORCIONAL POR ALOC_REC
# ============================================================

def executar_rateio_proporcional_aloc_rec(
    bd_demanda: pd.DataFrame,
    bd_alternativas_validas: pd.DataFrame
):
    '''Executa o balanceamento por rateio proporcional.

    Para cada nível de prioridade (PRIOR_MATPAR, PRIOR_ROT):
      1. Pega o saldo pendente dos itens com alternativa naquela prioridade.
      2. Agrupa por ALOC_REC.
      3. Se a demanda em horas do recurso couber na capacidade restante, atende tudo.
      4. Se estourar, aplica fator proporcional para todos os itens do recurso.
      5. O saldo não atendido do item segue para a próxima prioridade.
    '''
    demanda = bd_demanda.copy()
    alternativas = bd_alternativas_validas.copy()

    # Capacidade por ALOC_REC para a etapa ALOC_REC.
    # Nesta V1, a capacidade do recurso é obtida pelas capacidades agregadas das alternativas válidas.
    cap_aloc = (
        alternativas
        .groupby("ALOC_REC", as_index=False)
        .agg(
            HOR_REC=("HOR_REC", "max"),
            HOR_FER=("HOR_FER", "max")
        )
    )
    cap_aloc["HOR_CAP"] = cap_aloc[["HOR_REC", "HOR_FER"]].min(axis=1)

    capacidade_restante = cap_aloc.set_index("ALOC_REC")["HOR_CAP"].to_dict()
    pendente = demanda.set_index("ID_PROD_UNID_FAT")["NEC_PCS"].to_dict()

    producao = []

    niveis_prioridade = (
        alternativas[["PRIOR_MATPAR", "PRIOR_ROT"]]
        .drop_duplicates()
        .sort_values(["PRIOR_MATPAR", "PRIOR_ROT"])
        .itertuples(index=False, name=None)
    )

    for prior_matpar, prior_rot in niveis_prioridade:
        nivel = alternativas[
            alternativas["PRIOR_MATPAR"].eq(prior_matpar)
            & alternativas["PRIOR_ROT"].eq(prior_rot)
        ].copy()

        nivel["QTD_PENDENTE_ITEM"] = nivel["ID_PROD_UNID_FAT"].map(pendente).fillna(0)
        nivel = nivel[nivel["QTD_PENDENTE_ITEM"] > TOL].copy()

        if nivel.empty:
            continue

        # Se houver mais de uma alternativa no mesmo nível para o mesmo item,
        # divide a solicitação inicial entre elas para não duplicar demanda.
        qtd_alt_mesmo_nivel = nivel.groupby("ID_PROD_UNID_FAT")["ALOC_REC"].transform("count")
        nivel["QTD_SOLICITADA"] = nivel["QTD_PENDENTE_ITEM"] / qtd_alt_mesmo_nivel
        nivel["HR_SOLICITADA"] = nivel["QTD_SOLICITADA"] / nivel["PCS_HORA"]

        # A alternativa não pode solicitar mais horas do que sua própria HOR_CAP.
        nivel["HR_SOLICITADA"] = np.minimum(nivel["HR_SOLICITADA"], nivel["HOR_CAP"])
        nivel["QTD_SOLICITADA"] = nivel["HR_SOLICITADA"] * nivel["PCS_HORA"]

        for aloc_rec, grupo in nivel.groupby("ALOC_REC", sort=True):
            cap_disp = capacidade_restante.get(aloc_rec, 0)
            if cap_disp <= TOL:
                continue

            hr_solicitada_total = grupo["HR_SOLICITADA"].sum()
            if hr_solicitada_total <= TOL:
                continue

            fator_rateio = min(1.0, cap_disp / hr_solicitada_total)

            aloc = grupo.copy()
            aloc["FATOR_RATEIO_ALOC_REC"] = fator_rateio
            aloc["QTD_PRODUZIR_SOLVER"] = aloc["QTD_SOLICITADA"] * fator_rateio
            aloc["HR_PRODUZIR_SOLVER"] = aloc["HR_SOLICITADA"] * fator_rateio
            aloc = aloc[aloc["QTD_PRODUZIR_SOLVER"] > TOL].copy()

            if aloc.empty:
                continue

            producao.append(aloc)

            # Atualiza pendência por item
            produzido_item = aloc.groupby("ID_PROD_UNID_FAT")["QTD_PRODUZIR_SOLVER"].sum()
            for item, qtd in produzido_item.items():
                pendente[item] = max(0.0, pendente.get(item, 0) - float(qtd))

            # Atualiza capacidade restante do recurso
            capacidade_restante[aloc_rec] = max(
                0.0,
                cap_disp - float(aloc["HR_PRODUZIR_SOLVER"].sum())
            )

    if producao:
        bd_plano = pd.concat(producao, ignore_index=True)
    else:
        bd_plano = pd.DataFrame(columns=list(alternativas.columns) + [
            "QTD_PENDENTE_ITEM", "QTD_SOLICITADA", "HR_SOLICITADA",
            "FATOR_RATEIO_ALOC_REC", "QTD_PRODUZIR_SOLVER", "HR_PRODUZIR_SOLVER"
        ])

    return bd_plano, cap_aloc


bd_plano_producao, bd_cap_aloc_rec = executar_rateio_proporcional_aloc_rec(
    bd_demanda_solver,
    bd_alternativas_validas
)

print("Linhas plano produção:", len(bd_plano_producao))
print("ALOC_REC com capacidade:", len(bd_cap_aloc_rec))


Linhas plano produção: 1180
ALOC_REC com capacidade: 169


In [6]:

# ============================================================
# 05. SAÍDAS DE NEGÓCIO
# ============================================================

# 02_PLANO_PRODUCAO
cols_plano = [
    "MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD",
    "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID",
    "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP",
    "QTD_SOLICITADA", "HR_SOLICITADA", "FATOR_RATEIO_ALOC_REC",
    "QTD_PRODUZIR_SOLVER", "HR_PRODUZIR_SOLVER"
]

plano_producao = bd_plano_producao[[c for c in cols_plano if c in bd_plano_producao.columns]].copy()

# 03_NAO_ATEND_ITEM
atendido_item = (
    plano_producao
    .groupby(["MES_REF", "ID_PROD_UNID_FAT"], as_index=False)
    .agg(QTD_ATENDIDA_SOLVER=("QTD_PRODUZIR_SOLVER", "sum"))
    if not plano_producao.empty else
    pd.DataFrame(columns=["MES_REF", "ID_PROD_UNID_FAT", "QTD_ATENDIDA_SOLVER"])
)

nao_atend_item = bd_demanda_solver.merge(
    atendido_item,
    on=["MES_REF", "ID_PROD_UNID_FAT"],
    how="left"
)

nao_atend_item["QTD_ATENDIDA_SOLVER"] = nao_atend_item["QTD_ATENDIDA_SOLVER"].fillna(0)
nao_atend_item["QTD_NAO_ATEND_SOLVER"] = (
    nao_atend_item["NEC_PCS"] - nao_atend_item["QTD_ATENDIDA_SOLVER"]
).clip(lower=0)

cols_nao_atend = [
    "MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD",
    "NEC_PCS", "QTD_ATENDIDA_SOLVER", "QTD_NAO_ATEND_SOLVER"
]
nao_atend_item = nao_atend_item[[c for c in cols_nao_atend if c in nao_atend_item.columns]].copy()

# 01_RESUMO_MAQUINAS
prod_maquina = (
    plano_producao
    .groupby("ALOC_REC", as_index=False)
    .agg(
        HR_PRODUZIR_SOLVER=("HR_PRODUZIR_SOLVER", "sum"),
        QTD_PRODUZIR_SOLVER=("QTD_PRODUZIR_SOLVER", "sum"),
        QTD_ITENS=("ID_PROD_UNID_FAT", "nunique")
    )
    if not plano_producao.empty else
    pd.DataFrame(columns=["ALOC_REC", "HR_PRODUZIR_SOLVER", "QTD_PRODUZIR_SOLVER", "QTD_ITENS"])
)

resumo_maquinas = bd_cap_aloc_rec.merge(
    prod_maquina,
    on="ALOC_REC",
    how="left"
).fillna({
    "HR_PRODUZIR_SOLVER": 0,
    "QTD_PRODUZIR_SOLVER": 0,
    "QTD_ITENS": 0
})

resumo_maquinas["OCUPACAO_SOLVER_PCT"] = np.where(
    resumo_maquinas["HOR_CAP"] > 0,
    resumo_maquinas["HR_PRODUZIR_SOLVER"] / resumo_maquinas["HOR_CAP"],
    0
)
resumo_maquinas["HR_OCIOSA_SOLVER"] = resumo_maquinas["HOR_CAP"] - resumo_maquinas["HR_PRODUZIR_SOLVER"]
resumo_maquinas["ESTOURO_HR_SOLVER"] = np.where(
    resumo_maquinas["HR_PRODUZIR_SOLVER"] > resumo_maquinas["HOR_CAP"] + TOL,
    resumo_maquinas["HR_PRODUZIR_SOLVER"] - resumo_maquinas["HOR_CAP"],
    0
)

resumo_maquinas = resumo_maquinas[[
    "ALOC_REC", "HOR_REC", "HOR_FER", "HOR_CAP", "HR_PRODUZIR_SOLVER",
    "OCUPACAO_SOLVER_PCT", "HR_OCIOSA_SOLVER", "ESTOURO_HR_SOLVER",
    "QTD_PRODUZIR_SOLVER", "QTD_ITENS"
]].sort_values(["OCUPACAO_SOLVER_PCT", "HR_PRODUZIR_SOLVER"], ascending=[False, False])

# 04_AUDITORIA_CAPACIDADE
cols_auditoria = [
    "MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD",
    "PRIOR_MATPAR", "PRIOR_ROT", "ALOC_REC", "COD_FER_UNID", "HOR_REC", "HOR_FER", "HOR_CAP"
]
auditoria_capacidade = bd_alternativas_item[[c for c in cols_auditoria if c in bd_alternativas_item.columns]].copy()

# 05_ALTERNATIVAS_ITEM
cols_alternativas = [
    "MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD",
    "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID",
    "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP"
]
alternativas_item = bd_alternativas_item[[c for c in cols_alternativas if c in bd_alternativas_item.columns]].copy()

print("01_RESUMO_MAQUINAS:", resumo_maquinas.shape)
print("02_PLANO_PRODUCAO:", plano_producao.shape)
print("03_NAO_ATEND_ITEM:", nao_atend_item.shape)
print("04_AUDITORIA_CAPACIDADE:", auditoria_capacidade.shape)
print("05_ALTERNATIVAS_ITEM:", alternativas_item.shape)


01_RESUMO_MAQUINAS: (169, 10)
02_PLANO_PRODUCAO: (1180, 21)
03_NAO_ATEND_ITEM: (1144, 10)
04_AUDITORIA_CAPACIDADE: (2399, 14)
05_ALTERNATIVAS_ITEM: (2399, 16)


In [7]:

# ============================================================
# 06. VALIDAÇÕES
# ============================================================

nec_total = nao_atend_item["NEC_PCS"].sum()
atend_total = nao_atend_item["QTD_ATENDIDA_SOLVER"].sum()
nao_atend_total = nao_atend_item["QTD_NAO_ATEND_SOLVER"].sum()
dif_fechamento = nec_total - atend_total - nao_atend_total

qtd_itens = len(nao_atend_item)
qtd_itens_atendidos = (nao_atend_item["QTD_ATENDIDA_SOLVER"] > TOL).sum()
qtd_itens_nao_atendidos = (nao_atend_item["QTD_ATENDIDA_SOLVER"] <= TOL).sum()
qtd_itens_com_saldo = (nao_atend_item["QTD_NAO_ATEND_SOLVER"] > TOL).sum()

validacao = pd.DataFrame({
    "METRICA": [
        "NEC_PCS_TOTAL",
        "QTD_ATENDIDA_SOLVER_TOTAL",
        "QTD_NAO_ATEND_SOLVER_TOTAL",
        "DIF_FECHAMENTO_DEMANDA",
        "QTD_ITENS_DEMANDA",
        "QTD_ITENS_COM_PRODUCAO",
        "QTD_ITENS_SEM_PRODUCAO",
        "QTD_ITENS_COM_NAO_ATENDIMENTO",
        "QTD_ALOC_REC",
        "QTD_ALOC_REC_ESTOURO",
        "ESTOURO_HR_TOTAL",
        "HOR_CAP_TOTAL",
        "HR_PRODUZIR_SOLVER_TOTAL",
        "OCUPACAO_GLOBAL",
        "QTD_ALTERNATIVAS_AUDITADAS",
        "QTD_ALTERNATIVAS_VALIDAS_PRODUCAO",
        "QTD_ALTERNATIVAS_HOR_CAP_ZERO"
    ],
    "VALOR": [
        nec_total,
        atend_total,
        nao_atend_total,
        dif_fechamento,
        qtd_itens,
        qtd_itens_atendidos,
        qtd_itens_nao_atendidos,
        qtd_itens_com_saldo,
        len(resumo_maquinas),
        (resumo_maquinas["ESTOURO_HR_SOLVER"] > TOL).sum(),
        resumo_maquinas["ESTOURO_HR_SOLVER"].sum(),
        resumo_maquinas["HOR_CAP"].sum(),
        resumo_maquinas["HR_PRODUZIR_SOLVER"].sum(),
        resumo_maquinas["HR_PRODUZIR_SOLVER"].sum() / resumo_maquinas["HOR_CAP"].sum() if resumo_maquinas["HOR_CAP"].sum() > 0 else 0,
        len(bd_alternativas_item),
        len(bd_alternativas_validas),
        (bd_alternativas_item["HOR_CAP"] <= 0).sum()
    ]
})

display(validacao)


,METRICA,VALOR
0,NEC_PCS_TOTAL,5.859632e+07
1,QTD_ATENDIDA_SOLVER_TOTAL,3.400675e+07
2,QTD_NAO_ATEND_SOLVER_TOTAL,2.458957e+07
3,DIF_FECHAMENTO_DEMANDA,0.000000e+00
4,QTD_ITENS_DEMANDA,1.144000e+03
5,QTD_ITENS_COM_PRODUCAO,1.048000e+03
6,QTD_ITENS_SEM_PRODUCAO,9.600000e+01
7,QTD_ITENS_COM_NAO_ATENDIMENTO,5.000000e+02
8,QTD_ALOC_REC,1.690000e+02
9,QTD_ALOC_REC_ESTOURO,0.000000e+00


In [8]:

# ============================================================
# 07. EXPORTAÇÃO
# ============================================================

OUTPUT_SOLVER_FILE.parent.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(OUTPUT_SOLVER_FILE, engine="openpyxl") as writer:
    resumo_maquinas.to_excel(writer, sheet_name="01_RESUMO_MAQUINAS", index=False)
    plano_producao.to_excel(writer, sheet_name="02_PLANO_PRODUCAO", index=False)
    nao_atend_item.to_excel(writer, sheet_name="03_NAO_ATEND_ITEM", index=False)
    auditoria_capacidade.to_excel(writer, sheet_name="04_AUDITORIA_CAPACIDADE", index=False)
    alternativas_item.to_excel(writer, sheet_name="05_ALTERNATIVAS_ITEM", index=False)
    validacao.to_excel(writer, sheet_name="99_VALIDACAO", index=False)

print("Arquivo exportado:", OUTPUT_SOLVER_FILE)

wb = load_workbook(OUTPUT_SOLVER_FILE, read_only=True)
print("Guias exportadas:")
for s in wb.sheetnames:
    print("-", s)


Arquivo exportado: /mnt/data/bd_SOLVER_ALOC_REC_V1.xlsx
Guias exportadas:
- 01_RESUMO_MAQUINAS
- 02_PLANO_PRODUCAO
- 03_NAO_ATEND_ITEM
- 04_AUDITORIA_CAPACIDADE
- 05_ALTERNATIVAS_ITEM
- 99_VALIDACAO


In [9]:

# ============================================================
# 08. CHECK RÁPIDO DE UM PRODUTO ESPECÍFICO (OPCIONAL)
# ============================================================

# Altere o código abaixo se quiser investigar um produto específico.
COD_PROD_CHECK = "1236"

print("Plano de produção do produto:", COD_PROD_CHECK)
display(
    plano_producao[plano_producao["COD_PROD"].astype(str).str.zfill(4).eq(COD_PROD_CHECK.zfill(4))]
    .sort_values(["ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "ALOC_REC"])
)

print("Não atendimento do produto:", COD_PROD_CHECK)
display(
    nao_atend_item[nao_atend_item["COD_PROD"].astype(str).str.zfill(4).eq(COD_PROD_CHECK.zfill(4))]
    .sort_values(["ID_PROD_UNID_FAT"])
)

print("Alternativas do produto:", COD_PROD_CHECK)
display(
    alternativas_item[alternativas_item["COD_PROD"].astype(str).str.zfill(4).eq(COD_PROD_CHECK.zfill(4))]
    .sort_values(["ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "ALOC_REC"])
)


Plano de produção do produto: 1236


,MES_REF,ID_PROD_UNID_FAT,COD_PROD,DESC_PROD,UNID_PROD,UNID_FAT,TIPO_PROD,PRIOR_MATPAR,PRIOR_ROT,CLASS_ROT,ALOC_REC,COD_FER_UNID,PCS_HORA,HOR_REC,HOR_FER,HOR_CAP,QTD_SOLICITADA,HR_SOLICITADA,FATOR_RATEIO_ALOC_REC,QTD_PRODUZIR_SOLVER,HR_PRODUZIR_SOLVER
777,2026-06-01,JUN26|CENTRO-OESTE|1236,1236,ELETROD PVC FLEX CORRUG REFORCADO 25MM 50 MT,Centro-Oeste,Centro-Oeste,PA,1,1,2,EX507,1236B|CEN,1200,179.87,196.8,179.87,89950.0,74.958333,0.413781,37219.619353,31.016349
778,2026-06-01,JUN26|NORDESTE|1236,1236,ELETROD PVC FLEX CORRUG REFORCADO 25MM 50 MT,Centro-Oeste,Nordeste,PA,1,1,8,EX507,1236B|CEN,1200,179.87,196.8,179.87,215844.0,179.870000,0.413781,89312.190324,74.426825


Não atendimento do produto: 1236


,MES_REF,ID_PROD_UNID_FAT,COD_PROD,DESC_PROD,UNID_PROD,UNID_FAT,TIPO_PROD,NEC_PCS,QTD_ATENDIDA_SOLVER,QTD_NAO_ATEND_SOLVER
19,2026-06-01,JUN26|CENTRO-OESTE|1236,1236,ELETROD PVC FLEX CORRUG REFORCADO 25MM 50 MT,Centro-Oeste,Centro-Oeste,PA,89950.0,37219.619353,5.273038e+04
487,2026-06-01,JUN26|MATRIZ|1236,1236,ELETROD PVC FLEX CORRUG REFORCADO 25MM 50 MT,Matriz,Matriz,PA,1773200.0,0.000000,1.773200e+06
1003,2026-06-01,JUN26|NORDESTE|1236,1236,ELETROD PVC FLEX CORRUG REFORCADO 25MM 50 MT,Centro-Oeste,Nordeste,PA,776150.0,89312.190324,6.868378e+05


Alternativas do produto: 1236


,MES_REF,ID_PROD_UNID_FAT,COD_PROD,DESC_PROD,UNID_PROD,UNID_FAT,TIPO_PROD,PRIOR_MATPAR,PRIOR_ROT,CLASS_ROT,ALOC_REC,COD_FER_UNID,PCS_HORA,HOR_REC,HOR_FER,HOR_CAP
786,2026-06-01,JUN26|CENTRO-OESTE|1236,1236,ELETROD PVC FLEX CORRUG REFORCADO 25MM 50 MT,Centro-Oeste,Centro-Oeste,PA,1,1,2,EX507,1236B|CEN,1200,179.87,196.80,179.87
1425,2026-06-01,JUN26|CENTRO-OESTE|1236,1236,ELETROD PVC FLEX CORRUG REFORCADO 25MM 50 MT,Centro-Oeste,Centro-Oeste,PA,1,2,2,EX505,1236A|CEN,1200,89.77,196.80,89.77
1064,2026-06-01,JUN26|MATRIZ|1236,1236,ELETROD PVC FLEX CORRUG REFORCADO 25MM 50 MT,Matriz,Matriz,PA,1,1,4,26652,1236Q|MAT,2250,65.67,0.00,0.00
1647,2026-06-01,JUN26|MATRIZ|1236,1236,ELETROD PVC FLEX CORRUG REFORCADO 25MM 50 MT,Matriz,Matriz,PA,1,2,4,26651,1236U|MAT,2000,164.25,0.00,0.00
1808,2026-06-01,JUN26|MATRIZ|1236,1236,ELETROD PVC FLEX CORRUG REFORCADO 25MM 50 MT,Matriz,Matriz,PA,1,3,4,26503,1236T|MAT,1100,164.25,0.00,0.00
1881,2026-06-01,JUN26|MATRIZ|1236,1236,ELETROD PVC FLEX CORRUG REFORCADO 25MM 50 MT,Matriz,Matriz,PA,1,4,4,26506,1236V|MAT,1100,164.25,0.00,0.00
1126,2026-06-01,JUN26|NORDESTE|1236,1236,ELETROD PVC FLEX CORRUG REFORCADO 25MM 50 MT,Centro-Oeste,Nordeste,PA,1,1,8,EX507,1236B|CEN,1200,179.87,196.80,179.87
1702,2026-06-01,JUN26|NORDESTE|1236,1236,ELETROD PVC FLEX CORRUG REFORCADO 25MM 50 MT,Centro-Oeste,Nordeste,PA,1,2,8,EX505,1236A|CEN,1200,89.77,196.80,89.77
2143,2026-06-01,JUN26|NORDESTE|1236,1236,ELETROD PVC FLEX CORRUG REFORCADO 25MM 50 MT,Nordeste,Nordeste,PA,2,1,8,EX504,1236A|NOR,1350,168.90,172.62,168.90
2255,2026-06-01,JUN26|NORDESTE|1236,1236,ELETROD PVC FLEX CORRUG REFORCADO 25MM 50 MT,Nordeste,Nordeste,PA,2,2,8,EX601,1236A|NOR,1350,60.23,172.62,60.23
